[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/laurenceyoon/ksmpc2026/blob/main/notebooks/1_music_alignment.ipynb)

# KSMPC 2026 여름학교 — MIR 기반 서양음악 연주 분석의 기초

## 1. 음악 정렬 (Music Alignment) 실습

**KSMPC 2026 여름학교**의 *MIR 기반 서양음악 연주 분석의 기초* 세션 중 **음악 정렬** 실습 자료입니다.
같은 곡을 담은 음원, 악보, MIDI를 서로 **오프라인 정렬**하는 방법을 다룹니다.

---

### 음악 정렬이란

**같은 곡**을 담은 서로 다른 자료에서, 한쪽의 어느 지점이 다른 쪽의 어디에 해당하는지 찾는 것을 음악 정렬(music alignment)이라고 합니다. 연주 분석은 물론이고 score following, 자동 반주, annotation transfer가 모두 여기서 출발합니다.

이 노트북에서는 세 가지 정렬을 다룹니다.

| 과제 | 입력 | 도구 | 결과 |
|---|---|---|---|
| **Audio-to-Audio** | 연주 음원 2개 | [Sync Toolbox](https://github.com/meinardmueller/synctoolbox) | warping path (시간축 대응) |
| **Audio-to-Score** | 악보 + 연주 음원 | [Sync Toolbox](https://github.com/meinardmueller/synctoolbox) + [partitura](https://github.com/CPJKU/partitura) | 박(beat) → 초(second) 매핑 |
| **MIDI-to-Score** | 악보 + 연주 MIDI | [parangonar](https://github.com/sildater/parangonar) | 음표 단위 대응 (match/deletion/insertion) |

무엇을 단위로 정렬하느냐가 다릅니다. Sync Toolbox가 맞추는 것은 *시간축*입니다. 이 녹음의 12.4초가 저 녹음의 9.8초에 대응한다는 식입니다. parangonar는 *음표 하나하나*를 맞춥니다. 악보의 이 음표가 연주에서 어느 MIDI 음표로 나타났는지까지 알려주고, 연주자가 빠뜨린 음표와 악보에 없는 음표도 구분합니다.

### 사용하는 데이터

예시는 모두 `resources/` 폴더의 **쇼팽 에튀드 Op. 10 No. 3** 자료입니다.

- **연주 음원 2개** (`p09`, `p15`) — 같은 악보를 친 다른 연주입니다. 26.7초와 35.6초로 템포 차이가 큽니다
- **악보** — MusicXML과 이미지, 그리고 등속으로 렌더링한 MIDI
- **연주 MIDI 3개** — `2_automatic_music_transcription.ipynb`에서 자동 채보(AMT)로 만든 파일입니다

음원은 모두 **Vienna 4x22 Piano Corpus**에서 가져왔으며 `p09`, `p15`는 그중 9번, 15번 피아니스트의 연주입니다.

> 여기서 다루는 것은 **오프라인(비실시간) 정렬**입니다. 연주 전체를 미리 받아두고 정렬하므로 앞뒤를 모두 보고 판단할 수 있습니다. 연주가 흘러가는 도중에 실시간으로 위치를 따라가는 것은 **온라인(실시간) 정렬**으로 score following이라고도 합니다.

## 0. Environment Setup

아래 setup cell은 처음에 한 번만 실행하면 됩니다.

- Colab이면 저장소를 clone한 뒤 (실습에 쓸 음원·악보·MIDI가 들어 있습니다) package를 설치합니다.
- 로컬이면 이미 저장소 안에 있다고 보고 package만 설치합니다. conda를 쓴다면
  `conda create -n ksmpc python=3.11 && conda activate ksmpc`로 환경을 만들어 두세요.

이미 설치를 마쳤다면 설치 cell은 건너뛰어도 됩니다.

In [ ]:
#@title Setup: install dependencies (Colab + local) { display-mode: "form" }
import importlib.util
import shutil
import subprocess
import sys
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB and not Path("ksmpc2026").exists() and not Path("../resources").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/laurenceyoon/ksmpc2026.git"],
        check=True,
    )

if IN_COLAB and shutil.which("apt-get") is not None:
    # pyfluidsynth is only a binding; the native FluidSynth library and a
    # SoundFont have to come from apt.
    subprocess.run(["apt-get", "update", "-qq"], check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    subprocess.run(["apt-get", "install", "-y", "-qq", "fluidsynth", "fluid-soundfont-gm"],
                   check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

%pip install -q synctoolbox librosa parangonar partitura pyfluidsynth pandas matplotlib

%pip install -q --force-reinstall --no-deps "synctoolbox @ git+https://github.com/groupmm/synctoolbox.git"

print(f"Running in {'Colab' if IN_COLAB else 'a local environment'}. Installation complete.")

if IN_COLAB:
    # partitura caches HAS_FLUIDSYNTH at import time, so a session that imported it
    # before FluidSynth existed keeps failing. Restart to pick up the new libraries.
    print("Restarting the runtime. Afterwards, run the cells below (setup is done).")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)


In [ ]:
#@title Imports, parameters and resource paths { display-mode: "form" }
import importlib.util
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

import numpy as np
import pandas as pd
import librosa
import librosa.display
import partitura as pt
import matplotlib.pyplot as plt
import IPython.display as ipd

from synctoolbox.dtw.mrmsdtw import sync_via_mrmsdtw
from synctoolbox.dtw.utils import make_path_strictly_monotonic
from synctoolbox.feature.chroma import pitch_to_chroma, quantize_chroma
from synctoolbox.feature.pitch import audio_to_pitch_features
from synctoolbox.feature.utils import estimate_tuning
from synctoolbox.feature.visualization import plot_chromagram, plot_signal

%matplotlib inline

Fs = 22050              # sampling rate
feature_rate = 50       # feature frames per second
step_weights = np.array([1.5, 1.5, 2.0])
threshold_rec = 10 ** 6
figsize = (9, 3)

PROJECT_ROOT = next(
    (p.resolve() for p in (Path(".."), Path("ksmpc2026"), Path(".")) if (p / "resources").is_dir()),
    None,
)
assert PROJECT_ROOT is not None, "Could not find the repository's resources/ directory."
RESOURCE_DIR = PROJECT_ROOT / "resources"

PERFORMANCE_AUDIOS = {
    "p09": RESOURCE_DIR / "Chopin_op10_no3_p09_short.wav",
    "p15": RESOURCE_DIR / "Chopin_op10_no3_p15_short.wav",
}
SCORE_IMAGE = RESOURCE_DIR / "Chopin_op10_no3_p15_short.png"
SCORE_MUSICXML = RESOURCE_DIR / "Chopin_op10_no3_short.musicxml"
SCORE_MIDI = RESOURCE_DIR / "Chopin_op10_no3_short_score.mid"

PERFORMANCE_MIDIS = {
    "p09": RESOURCE_DIR / "Chopin_op10_no3_p09_short.mid",
    "p15": RESOURCE_DIR / "Chopin_op10_no3_p15_short.mid",
}
GROUND_TRUTH_MATCHES = {
    "p09": RESOURCE_DIR / "Chopin_op10_no3_p09.match",
    "p15": RESOURCE_DIR / "Chopin_op10_no3_p15.match",
}

OUTPUT_DIR = PROJECT_ROOT / "results" / "music_alignment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [*PERFORMANCE_AUDIOS.values(), SCORE_MUSICXML, SCORE_MIDI,
             *PERFORMANCE_MIDIS.values(), *GROUND_TRUTH_MATCHES.values()]:
    assert path.is_file(), f"Missing resource: {path}"
print(f"Setup complete. Resources found in {RESOURCE_DIR}")

# 1. Audio-to-Audio Alignment with Sync Toolbox

같은 곡이라도 연주마다 속도가 미묘하게 다릅니다. 한 연주를 다른 연주에 포개는 time-warping을 찾는 것이 alignment입니다.

쇼팽 에튀드 Op. 10 No. 3의 두 연주(`p09`, `p15`)로 Sync Toolbox pipeline을 따라갑니다.
tuning 추정 → chroma feature 계산 → MrMsDTW 정렬.

In [ ]:
display(ipd.Image(filename=str(SCORE_IMAGE), width=900))

audios = {}
for name, path in PERFORMANCE_AUDIOS.items():
    audio, _ = librosa.load(path, sr=Fs)
    audios[name] = audio
    print(f"{name}: {path.name} ({len(audio) / Fs:.1f} seconds)")
    plot_signal(audio, Fs=Fs, ylabel="Amplitude", title=f"Performance {name}", figsize=figsize)
    plt.show()
    display(ipd.Audio(audio, rate=Fs))

audio_1, audio_2 = audios["p09"], audios["p15"]

## 1.1 Estimating Tuning

녹음은 A=440 Hz에서 조금씩 어긋나 있는 경우가 많습니다. 녹음마다 이 편차를 추정해 feature extraction에 넘깁니다. 보정하지 않으면 chroma feature가 인접 pitch class로 번져 alignment 정확도가 떨어질 수 있습니다.

In [ ]:
tuning_offset_1 = estimate_tuning(audio_1, Fs)
tuning_offset_2 = estimate_tuning(audio_2, Fs)
print(f"Estimated tuning deviation - p09: {tuning_offset_1} cents, p15: {tuning_offset_2} cents")

## 1.2 Computing Chroma Features

Chroma feature는 spectrum을 12개 pitch class로 접습니다. 음색과 옥타브를 버리고 화성만 남기므로, 악기와 녹음 환경이 다른 두 연주도 비교할 수 있습니다.

smoothing, downsampling, normalization은 MrMsDTW가 내부에서 처리하므로 quantized chroma를 그대로 넘깁니다.

In [ ]:
def get_chroma_from_audio(audio, tuning_offset, visualize=False):
    """Compute quantized chroma features for one audio signal."""
    f_pitch = audio_to_pitch_features(
        f_audio=audio,
        Fs=Fs,
        tuning_offset=tuning_offset,
        feature_rate=feature_rate,
        verbose=visualize,
    )
    f_chroma = pitch_to_chroma(f_pitch=f_pitch)
    return quantize_chroma(f_chroma=f_chroma)


f_chroma_quantized_1 = get_chroma_from_audio(audio_1, tuning_offset_1)
f_chroma_quantized_2 = get_chroma_from_audio(audio_2, tuning_offset_2)

plot_chromagram(f_chroma_quantized_1, Fs=feature_rate, title="Chroma - performance p09", figsize=figsize)
plt.show()
plot_chromagram(f_chroma_quantized_2, Fs=feature_rate, title="Chroma - performance p15", figsize=figsize)
plt.show()

## 1.3 Aligning with MrMsDTW

**MrMsDTW**(multi-resolution multi-scale DTW)는 거친 해상도에서 시작해 점점 세밀하게 정렬합니다. 각 단계는 앞 단계가 찾은 alignment 근처만 탐색하므로 N × M cost matrix를 전부 채울 필요가 없습니다. 곡 전체를 넣어도 빠르고 메모리를 적게 쓰는 장점이 있습니다.

결과는 **warping path** `wp`입니다. `[n, m]`은 p09의 chroma frame *n*이 p15의 chroma frame *m*에 대응한다는 뜻이며, feature rate가 50 fps이므로 `feature_rate`로 나누면 초 단위가 됩니다.

In [ ]:
wp = sync_via_mrmsdtw(
    f_chroma1=f_chroma_quantized_1,
    f_chroma2=f_chroma_quantized_2,
    input_feature_rate=feature_rate,
    step_weights=step_weights,
    threshold_rec=threshold_rec,
    verbose=True,
)

n_levels = len(plt.get_fignums())
for level, num in enumerate(plt.get_fignums()):
    ax = plt.figure(num).axes[0]
    if level == n_levels - 1:
        unit = f"frames @ {feature_rate} fps"
    else:
        unit = f"downsampled frames, level {level + 1}"
    ax.set_ylabel(f"p09 ({unit})")
    ax.set_xlabel(f"p15 ({unit})")
plt.show()

### Making the Warping Path Strictly Monotonic

기본 DTW step size는 수평·수직 이동을 허용합니다. 그래서 raw path에는 한쪽이 진행되는 동안 다른 쪽이 멈춰 있는 구간이 생깁니다. 두 timeline 사이로 annotation을 옮길 때는 이런 구간이 문제가 되므로, **strictly monotonic** path로 다듬어 선형 보간으로 메웁니다.

In [ ]:
print(f"Warping path length (raw): {wp.shape[1]}")
wp = make_path_strictly_monotonic(wp)
print(f"Warping path length (strictly monotonic): {wp.shape[1]}")

### Plotting the Warping Path

Warping path를 초 단위로 그립니다. 대각선이면 두 연주의 tempo가 같다는 뜻이고, 대각선에서 벗어난 곳이 timing이 어긋난 지점입니다.

아래 panel은 path를 따라 누적된 두 연주의 시간 차이입니다.

In [ ]:
wp_seconds = wp / feature_rate
t_p09, t_p15 = wp_seconds[0], wp_seconds[1]

fig, axes = plt.subplots(2, 1, figsize=(7, 8), height_ratios=[3, 1], constrained_layout=True)

axes[0].plot(t_p09, t_p15, color="crimson", linewidth=2, label="Warping path")
diagonal = [0, min(t_p09[-1], t_p15[-1])]
axes[0].plot(diagonal, diagonal, color="gray", linestyle="--", linewidth=1, label="Identical tempo")
axes[0].set_xlabel("Time in p09 (seconds)")
axes[0].set_ylabel("Time in p15 (seconds)")
axes[0].set_title("Warping path between the two performances")
axes[0].legend()
axes[0].grid(alpha=0.2)
axes[0].set_aspect("equal")

axes[1].plot(t_p09, t_p15 - t_p09, color="royalblue", linewidth=2)
axes[1].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[1].set_xlabel("Time in p09 (seconds)")
axes[1].set_ylabel("p15 - p09 (s)")
axes[1].set_title("Accumulated time difference")
axes[1].grid(alpha=0.2)
plt.show()

print(f"Total duration - p09: {len(audio_1) / Fs:.1f} s, p15: {len(audio_2) / Fs:.1f} s")

## 1.4 Exporting the Alignment

Warping path를 notebook 밖에서도 쓸 수 있게 저장해 둡니다.

- **`warping_path_p09_p15.csv`** — 두 녹음의 대응 시각을 담은 전체 path
- **`sonic_visualiser_*.csv`** — [Sonic Visualiser](https://www.sonicvisualiser.org/)용 correspondence layer.
  한쪽 녹음을 연 뒤 *File > Import Annotation Layer...* 에서 해당 CSV를 선택하면 됩니다. 각 point에는 상대편 녹음의 대응 시각이 label로 붙어 있습니다.

Colab에서는 이 파일들이 runtime 안에 저장되며 session이 끝나면 사라집니다. 로컬에서는 `results/music_alignment/`에 그대로 남습니다.

In [ ]:
pd.DataFrame({"time_p09_sec": t_p09, "time_p15_sec": t_p15}).to_csv(
    OUTPUT_DIR / "warping_path_p09_p15.csv", index=False
)
print(f"Wrote warping_path_p09_p15.csv with {wp.shape[1]} points")

sv_interval_sec = 1.0
for name, (own, other) in {"p09": (t_p09, t_p15), "p15": (t_p15, t_p09)}.items():
    grid = np.arange(own[0], own[-1], sv_interval_sec)
    pd.DataFrame({
        "time": grid,
        "label": np.round(np.interp(grid, own, other), 3),
    }).to_csv(OUTPUT_DIR / f"sonic_visualiser_{name}.csv", index=False, header=False)
    print(f"Wrote sonic_visualiser_{name}.csv")

if IN_COLAB:
    from google.colab import files
    for path in sorted(OUTPUT_DIR.glob("*.csv")):
        files.download(str(path))

# 2. Audio-to-Score Alignment

**녹음**과 **악보**도 같은 방식으로 정렬합니다. 악보를 audio로 rendering하면 앞에서 푼 audio-to-audio 문제가 되기 때문입니다. rendering된 악보는 음표가 몇 초에 오는지 정확히 아는 기준 timeline 역할을 합니다.

결과는 기보된 beat가 이 녹음에서 몇 초에 들리는지의 대응표입니다. 악보 쪽 annotation을 녹음 위로 옮기거나, 마디별 템포를 재거나, 여러 연주를 악보 좌표에서 비교할 때 씁니다.


## 2.1 Reading the Score with partitura

악보는 **MusicXML**을 [partitura](https://github.com/CPJKU/partitura)로 읽습니다. partitura는 기보를 `note_array` 구조로 파싱하며, MIDI에는 남지 않는 *기보상의* 정보를 그대로 유지합니다. onset이 초가 아닌 **beat** 단위이고, note id와 voice, 기보된 길이가 함께 들어 있습니다.

기보는 음표 사이의 **상대적인** 길이 관계만 규정합니다. tempo marking이 있어도(이 곡은 *Lento, ma non troppo*) 대략적인 지시일 뿐, 연주자의 rubato와 해석에 따라 같은 beat가 매번 다른 길이로 실현됩니다. 그 beat가 이 연주에서 몇 초였는지 확정하는 것이 alignment입니다. 아래는 도입부(pickup 8분음표와 첫 마디)입니다.

In [ ]:
score = pt.load_score(str(SCORE_MUSICXML))
score_na = pt.utils.music.ensure_notearray(score)
score_positions = np.unique(score_na["onset_beat"])

print(f"Number of notes: {len(score_na)}")
print(f"Number of unique score onset positions: {len(score_positions)}")

display(ipd.Image(filename=str(SCORE_IMAGE), width=900))

opening = pd.DataFrame(score_na)[["id", "pitch", "onset_beat", "duration_beat", "voice"]]
display(opening[opening["onset_beat"] < 2.0].sort_values("onset_beat"))

NameError: name 'pt' is not defined

## 2.2 Rendering the Score to Audio

악보를 audio로 만들면 섹션 1의 pipeline을 그대로 쓸 수 있습니다. deadpan MIDI rendering은 모든 음표를 일정한 tempo와 velocity로 연주하므로 그 자체가 기준 timeline이 됩니다.

합성에는 partitura의 `save_wav_fluidsynth`를 씁니다. MuseScore General soundfont를 자동으로 내려받아 피아노 음색으로 렌더링하므로, 사인파 합성보다 실제 녹음에 가까운 소리가 납니다.

In [ ]:
score_performance = pt.load_performance_midi(str(SCORE_MIDI))

# partitura decides HAS_FLUIDSYNTH once, at import time. If this session imported
# partitura before FluidSynth was installed, that stale False survives re-imports,
# so refresh it here instead of forcing a restart.
import importlib
import partitura.utils.fluidsynth as _pt_fs
import partitura.io.exportaudio as _pt_audio

if not _pt_fs.HAS_FLUIDSYNTH:
    importlib.reload(_pt_fs)
    importlib.reload(_pt_audio)
assert _pt_fs.HAS_FLUIDSYNTH, (
    "FluidSynth is still unavailable. Run the setup cell, then restart the runtime "
    "(Runtime > Restart session) and run the cells again."
)

# partitura ships a SoundFont; fall back to the system one (apt fluid-soundfont-gm).
soundfont = Path(_pt_fs.DEFAULT_SOUNDFONT)
if not soundfont.exists():
    candidates = sorted(Path("/usr/share/sounds").rglob("*.sf[23]"))
    assert candidates, "No SoundFont found. Re-run the setup cell."
    soundfont = candidates[0]

score_audio = np.asarray(
    _pt_audio.save_wav_fluidsynth(
        score_performance, out=None, samplerate=Fs, soundfont=str(soundfont)
    )
).astype(np.float32)
score_audio /= np.abs(score_audio).max()
performance_audio = audios["p15"]

print(f"Score rendering: {len(score_audio) / Fs:.1f} seconds")
display(ipd.Audio(score_audio, rate=Fs, normalize=True))
print(f"Performance p15: {len(performance_audio) / Fs:.1f} seconds")
display(ipd.Audio(performance_audio, rate=Fs, normalize=True))


## 2.3 Aligning the Score Rendering to the Recording

In [ ]:
f_chroma_score = get_chroma_from_audio(score_audio, estimate_tuning(score_audio, Fs))
f_chroma_perf = get_chroma_from_audio(performance_audio, estimate_tuning(performance_audio, Fs))

plot_chromagram(f_chroma_score, Fs=feature_rate, title="Chroma - score rendering", figsize=figsize)
plt.show()
plot_chromagram(f_chroma_perf, Fs=feature_rate, title="Chroma - performance p15", figsize=figsize)
plt.show()

Rendering된 악보 쪽 chroma가 훨씬 깨끗합니다. pedal로 번지지 않고, 공간 울림도 없고, velocity도 일정합니다.
그래도 alignment는 이 차이에 영향받지 않습니다. chroma는 어떤 pitch class가 울리는지만 볼 뿐 그것이 *어떻게* 들리는지는 보지 않기 때문입니다.

In [ ]:
wp_score = sync_via_mrmsdtw(
    f_chroma1=f_chroma_score,
    f_chroma2=f_chroma_perf,
    input_feature_rate=feature_rate,
    step_weights=step_weights,
    threshold_rec=threshold_rec,
    verbose=False,
)
wp_score = make_path_strictly_monotonic(wp_score)
wp_score_seconds = wp_score / feature_rate
t_score, t_perf = wp_score_seconds[0], wp_score_seconds[1]
print(f"Score-to-performance warping path: {wp_score.shape[1]} points")

## 2.4 Transferring Score Positions onto the Recording

Warping path는 rendering된 악보의 시각을 연주의 시각으로 바꿉니다. 여기에 deadpan rendering의 beat ↔ 초 대응을 이어 붙이면 우리가 원하던 답을 얻습니다. **기보된 beat가 이 녹음에서 몇 초에 들리는가**입니다.

    beat (score) -> second (rendering) -> second (performance)


In [ ]:
render_na = pt.load_performance_midi(str(SCORE_MIDI)).note_array()
render_onsets_sec = np.unique(render_na["onset_sec"])
assert len(render_onsets_sec) == len(score_positions), "rendering and score positions disagree"

positions_in_performance = np.interp(render_onsets_sec, t_score, t_perf)

beat_map = pd.DataFrame({
    "onset_beat": score_positions,
    "rendering_sec": render_onsets_sec,
    "performance_sec": np.round(positions_in_performance, 3),
})
print(f"Transferred {len(score_positions)} score positions onto the performance timeline.")
display(beat_map.head(8))

predicted_onsets_sec = np.interp(score_na["onset_beat"], score_positions, positions_in_performance)
print(f"...covering all {len(score_na)} notated notes.")

옮겨 온 onset을 연주 waveform 위에 표시하고, 같은 위치에 click을 섞어 들려줍니다. alignment가 정확하면 click이 음표 위에 떨어집니다.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), constrained_layout=True)

axes[0].plot(t_score, t_perf, color="crimson", linewidth=2, label="Warping path")
diagonal = [0, min(t_score[-1], t_perf[-1])]
axes[0].plot(diagonal, diagonal, color="gray", linestyle="--", linewidth=1, label="Identical tempo")
axes[0].set_xlabel("Time in score rendering (seconds)")
axes[0].set_ylabel("Time in performance p15 (seconds)")
axes[0].set_title("Score-to-performance warping path")
axes[0].legend()
axes[0].grid(alpha=0.2)

times = np.arange(len(performance_audio)) / Fs
axes[1].plot(times, performance_audio, color="lightsteelblue", linewidth=0.5)
axes[1].vlines(predicted_onsets_sec, -1, 1, color="crimson", alpha=0.5, linewidth=0.8)
axes[1].set_xlim(0, times[-1])
axes[1].set_xlabel("Time (seconds)")
axes[1].set_ylabel("Amplitude")
axes[1].set_title("Performance p15 with transferred score onsets")
plt.show()

clicks = librosa.clicks(times=predicted_onsets_sec, sr=Fs, length=len(performance_audio))
print("Performance with transferred score onsets as clicks")
display(ipd.Audio(performance_audio + 0.5 * clicks, rate=Fs, normalize=True))

# 3. MIDI-to-Score Alignment with parangonar

[**parangonar**](https://github.com/sildater/parangonar)는 symbolic music alignment를 위한 Python library입니다. 오스트리아 JKU와 빈 국립음대(MDW)에서 개발했으며, `pip install parangonar`로 설치합니다.
악보와 연주를 note 단위로 정렬하는 여러 matching algorithm과 함께, 결과를 평가·시각화·저장하는 도구를 제공합니다. 앞서 쓴 [partitura](https://github.com/CPJKU/partitura)가 악보와 연주를 읽어 `note_array`로 만들어 주면, parangonar가 그 둘을 짝지어 주는 구조입니다.

Sync Toolbox가 맞추는 것이 *timeline*이라면, parangonar가 맞추는 것은 *note event*입니다. 악보의 음표 하나하나를 연주 MIDI에서 그것을 실현한 음표와 짝지어 주고, 짝이 없는 음표도 빠짐없이 짚어 줍니다.

time warping보다 까다로운 문제입니다. 연주자는 화음을 arpeggio로 펼치고, 장식음을 넣고, 음표를 빠뜨리거나 더하기도 합니다. 시간 순서대로 하나씩 대응시킬 수 없다는 뜻입니다. parangonar는 음표마다 다음 셋 중 하나를 붙입니다.

- **match** — 악보 음표와 연주 음표가 짝지어짐
- **deletion** — 악보에 있는데 연주되지 않음
- **insertion** — 연주에 있는데 악보에 없음

여기 쓰는 연주 MIDI는 Vienna 4x22 corpus의 실측 데이터입니다. 뵈젠도르퍼 컴퓨터 제어 피아노가 연주를 그대로 기록한 것이라 채보 오류가 없고, 따라서 insertion과 deletion은 온전히 연주자의 선택을 반영합니다.

### 정답 alignment 불러오기

Vienna 4x22 corpus에는 연주 MIDI뿐 아니라 사람이 검수한 **정답 alignment**가 `.match` 파일로 함께 들어 있습니다. 덕분에 matcher의 결과를 눈으로만 보는 대신 수치로 평가할 수 있습니다.

다만 이 `.match` 파일은 곡 전체(454 음표)를 담고 있고 우리가 쓰는 것은 앞부분 발췌입니다. 그래서 발췌의 beat 범위에 해당하는 음표만 남기고 잘라냅니다. 악보와 연주 note array도 `.match` 파일에 실린 것을 그대로 쓰는데, 이렇게 해야 정답과 예측이 같은 note id 체계 위에서 비교됩니다.

In [ ]:
import parangonar as pa

print(f"partitura {pt.__version__}, parangonar {pa.__version__}")

EXCERPT_LAST_BEAT = score_na["onset_beat"].max()

score_arrays, performances, ground_truths = {}, {}, {}
performance_objects, score_objects = {}, {}
for name, match_path in GROUND_TRUTH_MATCHES.items():
    performance, alignment, match_score = pt.load_match(str(match_path), create_score=True)
    full_score_na = pt.utils.music.ensure_notearray(match_score)

    excerpt_na = full_score_na[full_score_na["onset_beat"] <= EXCERPT_LAST_BEAT]
    excerpt_score_ids = {str(i) for i in excerpt_na["id"]}
    trimmed = [a for a in alignment if str(a.get("score_id")) in excerpt_score_ids]
    kept_performance_ids = {
        str(a["performance_id"]) for a in trimmed if a["label"] == "match"
    }

    performance_na = performance.note_array()
    performance_na = performance_na[
        [str(i) in kept_performance_ids for i in performance_na["id"]]
    ]

    score_arrays[name] = excerpt_na
    performances[name] = performance_na
    ground_truths[name] = trimmed
    performance_objects[name] = performance
    score_objects[name] = match_score

    labels = pd.Series([a["label"] for a in trimmed]).value_counts()
    print(f"{name}: score {len(excerpt_na)} notes, performance {len(performance_na)} notes, "
          f"ground truth {dict(labels)}")

## 3.1 Running the Note Matcher

`AutomaticNoteMatcher`는 anchor point를 직접 지정하지 않아도 note 단위로 정렬합니다. 두 sequence를 먼저 개략적으로 맞춘 뒤, 그 구간 안에서 음표를 하나씩 짝짓습니다.

In [ ]:
matcher = pa.AutomaticNoteMatcher()

alignments = {}
for name, performance_na in performances.items():
    alignment = matcher(score_arrays[name], performance_na)
    alignments[name] = alignment

    labels = pd.Series([note["label"] for note in alignment]).value_counts()
    matched = labels.get("match", 0)
    print(f"{name}: {matched} matches, "
          f"{labels.get('deletion', 0)} deletions, {labels.get('insertion', 0)} insertions "
          f"({matched / len(score_arrays[name]):.1%} of the score matched)")

## 3.2 Visualizing the Note Alignment

악보 음표와 연주 음표를 각각 piano roll로 그리고 짝지어진 것끼리 선으로 잇습니다. 선의 기울기가 그 지점의 tempo 관계를 나타내며, 선이 닿지 않은 음표가 insertion과 deletion입니다.

In [ ]:
target = "p15"
pa.plot_alignment(
    performances[target],
    score_arrays[target],
    alignments[target],
    fname=f"Note alignment - score vs {target}",
)
plt.show()

## 3.3 Evaluating Against the Ground Truth

이제 matcher가 찾은 alignment를 정답과 대조합니다. matched pair를 기준으로 precision, recall, F-score를 계산합니다.

- **precision** — matcher가 짝지은 것 중 정답에도 있는 비율
- **recall** — 정답의 짝 중 matcher가 찾아낸 비율
- **F-score** — 둘의 조화평균

In [ ]:
rows = []
for name, alignment in alignments.items():
    precision, recall, f_score = pa.fscore_alignments(
        alignment, ground_truths[name], types=["match"]
    )
    rows.append({
        "performance": name,
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "F-score": round(f_score, 4),
    })
display(pd.DataFrame(rows))

두 연주 모두 F-score가 1.0입니다. `AutomaticNoteMatcher`가 정답 alignment를 완벽하게 복원했다는 뜻입니다.

이 발췌는 소리가 깨끗하고 반복 구조가 없어 matcher에게 유리한 조건입니다. 곡이 길어지고 반복이 늘어나거나 연주가 악보에서 크게 벗어나면 점수는 떨어집니다. 정답이 있는 데이터로 이렇게 수치를 확인해 두어야, 정답이 없는 실제 데이터에 적용할 때 결과를 어느 정도 신뢰할지 판단할 수 있습니다.

## 3.4 Extracting Performance Timing

음표 단위로 짝이 맞춰져 있으므로 match된 음표마다 기보된 onset과 실제 연주 시각을 나란히 볼 수 있습니다.
기보된 beat와 연주 시각의 이 대응표가 연주 분석의 출발점입니다.

In [ ]:
def alignment_to_dataframe(alignment, score_note_array, performance_note_array):
    """Collect matched note pairs into a DataFrame of notated vs. performed timing."""
    score_by_id = {str(note["id"]): note for note in score_note_array}
    performance_by_id = {str(note["id"]): note for note in performance_note_array}

    rows = []
    for note in alignment:
        if note["label"] != "match":
            continue
        score_note = score_by_id.get(str(note["score_id"]))
        performance_note = performance_by_id.get(str(note["performance_id"]))
        if score_note is None or performance_note is None:
            continue
        rows.append({
            "score_id": str(note["score_id"]),
            "onset_beat": score_note["onset_beat"],
            "pitch": performance_note["pitch"],
            "onset_sec": performance_note["onset_sec"],
            "duration_sec": performance_note["duration_sec"],
            "velocity": performance_note["velocity"],
        })
    return pd.DataFrame(rows).sort_values("onset_beat").reset_index(drop=True)


timing = alignment_to_dataframe(
    alignments[target], score_arrays[target], performances[target]
)
display(timing.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
scatter = ax.scatter(
    timing["onset_beat"], timing["onset_sec"],
    c=timing["velocity"], cmap="magma", s=25,
)
ax.set_xlabel("Notated onset (beats)")
ax.set_ylabel("Performed onset (seconds)")
ax.set_title(f"Notated versus performed onsets - {target}")
ax.grid(alpha=0.2)
fig.colorbar(scatter, ax=ax, label="MIDI velocity")
plt.show()

이 곡선의 기울기가 그 지점의 tempo입니다. 가파른 구간은 느리게, 완만한 구간은 빠르게 연주한 부분입니다.
직선에서 벗어난 정도가 연주자의 rubato입니다.

## 3.5 Exporting the Alignment

parangonar는 이 분야에서 통용되는 형식으로 alignment를 저장합니다. `match` 파일이 note alignment의 표준 형식이며, 앞서 정답으로 쓴 `resources/*.match`도 같은 형식입니다. Parangonada CSV 묶음은 [https://sildater.github.io/parangonada/](https://sildater.github.io/parangonada/)에 올려 브라우저에서 확인할 수 있습니다.

In [ ]:
for name, alignment in alignments.items():
    export_dir = OUTPUT_DIR / name.replace(" ", "_").replace("(", "").replace(")", "")
    export_dir.mkdir(parents=True, exist_ok=True)

    pt.io.exportparangonada.save_parangonada_csv(
        alignment,
        performance_objects[name],
        score_objects[name],
        outdir=str(export_dir),
    )
    pt.save_match(
        alignment=alignment,
        performance_data=performance_objects[name],
        score_data=score_objects[name],
        out=str(export_dir / "alignment.match"),
    )
    print(f"{name} -> {export_dir}")

## 3.6 Collecting the Results

이 notebook이 만든 파일을 모두 나열합니다. Colab으로 실행 중이면 전체를 zip으로 묶어 자동으로 내려받습니다.

In [ ]:
print(f"Generated files in {OUTPUT_DIR}:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(OUTPUT_DIR)} ({path.stat().st_size:,} bytes)")

if IN_COLAB:
    import shutil
    from google.colab import files

    archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
    print(f"Downloading {Path(archive).name}")
    files.download(archive)

# 4. Try It Yourself — Use Your Own Recording

직접 가져온 녹음으로 정렬해 봅니다. **같은 곡의 서로 다른 연주 2개**를 올리면 섹션 1과 같은 pipeline으로 정렬합니다.

아래 cell을 실행하면 파일 선택 버튼이 나타납니다. 파일을 **2개** 고르세요. `.wav`, `.mp3`, `.flac` 등 대부분의 형식을 지원합니다.

> **참고.** 30초 안팎으로 자른 excerpt가 결과를 보기 좋습니다. 두 녹음이 **같은 구간**이어야 정렬이 의미를 갖습니다.

In [ ]:
MY_AUDIO_PATHS = []

if IN_COLAB:
    from google.colab import files
    print("같은 곡의 서로 다른 연주 2개를 선택하세요.")
    uploaded = files.upload()
    MY_AUDIO_PATHS = [Path(name) for name in uploaded]
else:
    # MY_AUDIO_PATHS = [Path("my_performance_1.wav"), Path("my_performance_2.wav")]
    print("로컬 환경입니다. MY_AUDIO_PATHS에 파일 경로를 직접 지정하세요.")

print()
print(f"선택된 파일 {len(MY_AUDIO_PATHS)}개:")
for path in MY_AUDIO_PATHS:
    print("  -", path)

올린 두 파일을 섹션 1과 같은 순서(tuning → chroma → MrMsDTW)로 정렬합니다.

In [ ]:
if len(MY_AUDIO_PATHS) != 2:
    print(f"연주 2개가 필요합니다. 현재 {len(MY_AUDIO_PATHS)}개가 지정되어 있습니다.")
else:
    my_audio_1, _ = librosa.load(MY_AUDIO_PATHS[0], sr=Fs)
    my_audio_2, _ = librosa.load(MY_AUDIO_PATHS[1], sr=Fs)
    print(f"{MY_AUDIO_PATHS[0].name}: {len(my_audio_1) / Fs:.1f}초")
    display(ipd.Audio(my_audio_1, rate=Fs))
    print(f"{MY_AUDIO_PATHS[1].name}: {len(my_audio_2) / Fs:.1f}초")
    display(ipd.Audio(my_audio_2, rate=Fs))

    my_chroma_1 = get_chroma_from_audio(my_audio_1, estimate_tuning(my_audio_1, Fs))
    my_chroma_2 = get_chroma_from_audio(my_audio_2, estimate_tuning(my_audio_2, Fs))
    my_wp = sync_via_mrmsdtw(
        f_chroma1=my_chroma_1,
        f_chroma2=my_chroma_2,
        input_feature_rate=feature_rate,
        step_weights=step_weights,
        threshold_rec=threshold_rec,
        verbose=False,
    )
    my_wp = make_path_strictly_monotonic(my_wp) / feature_rate

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(my_wp[0], my_wp[1], color="crimson", linewidth=2, label="Warping path")
    diagonal = [0, min(my_wp[0][-1], my_wp[1][-1])]
    ax.plot(diagonal, diagonal, color="gray", linestyle="--", linewidth=1, label="동일 tempo")
    ax.set_xlabel(f"{MY_AUDIO_PATHS[0].name} (초)")
    ax.set_ylabel(f"{MY_AUDIO_PATHS[1].name} (초)")
    ax.set_title("내 녹음 2개의 Warping path")
    ax.legend()
    ax.grid(alpha=0.2)
    ax.set_aspect("equal")
    plt.show()

# References

[1] Meinard Müller, Yigitcan Özer, Michael Krause, Thomas Prätzlich, and Jonathan Driedger: Sync Toolbox: A Python Package for Efficient, Robust, and Accurate Music Synchronization, JOSS, 2021.

[2] Thomas Prätzlich, Jonathan Driedger, and Meinard Müller: Memory-Restricted Multiscale Dynamic Time Warping, ICASSP, 2016.

[3] Meinard Müller, Henning Mattes, and Frank Kurth: An Efficient Multiscale Approach to Audio Synchronization, ISMIR, 2006.

[4] Silvan David Peter, Carlos Cancino-Chacón, Francesco Foscarin, Andrew McLeod, Florian Henkel, Emmanouil Karystinaios, and Gerhard Widmer: Automatic Note-Level Score-to-Performance Alignments in the ASAP Dataset, TISMIR, 2023.

[5] Carlos Cancino-Chacón, Silvan David Peter, Emmanouil Karystinaios, Francesco Foscarin, Maarten Grachten, and Gerhard Widmer: Partitura: A Python Package for Symbolic Music Processing, MEC, 2022.

[6] Werner Goebl: The Vienna 4x22 Piano Corpus, mdw – University of Music and Performing Arts Vienna, 1999.
https://doi.org/10.21939/4X22